In [1]:
!pip install stochastic


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functions import *
import warnings
from datetime import datetime

# optionally import stochastic if you intend to use it
try:
    from stochastic.processes.continuous import GeometricBrownianMotion
except ImportError:
    GeometricBrownianMotion = None

In [6]:
data = pd.read_csv(f'../{fullDataPath('BTC')}')

FileNotFoundError: [Errno 2] No such file or directory: '../fulldata/BTC_df.csv'

In [ ]:
# Convert time to datetime if it's not already
data['time'] = pd.to_datetime(data['time'], errors='coerce')
data = data.dropna(subset=['time'])

# Extract date component
data['date'] = data['time'].dt.date

# Group by date and aggregate values (one row per day)
# Hard coded change later
daily_data = (
    data
    .groupby('date')
    .agg(
        close=('close', 'last'),  # Take the last close price of the day
        avg_sentiment=('score', 'mean'),  # Average sentiment for the day
        tweet_count=('score', 'count'),   # Number of observations per day
        volume=('volume', 'last'),          # Total volume for the day
        BB_Lower=('BB_Lower', 'last'),  # Last value of BB lower band
        BB_Middle=('BB_Middle', 'last'),  # Last value of BB middle band
        BB_Upper=('BB_Upper', 'last'),  # Last value of BB upper band
        SMA_50=('SMA_50', 'last'),  # Last value of SMA 50
        Volume_MA_20=('Volume_MA_20', 'last'),  # Last value of Volume MA 20
        value=('value', 'last'),
        value_classification=('value_classification', 'last'),
        OBV=('OBV', 'last')
    )
    .reset_index()
)

# Convert date back to datetime format if needed
daily_data['time'] = pd.to_datetime(daily_data['date'])
daily_data = daily_data.drop('date', axis=1)

# Sort by time
daily_data = daily_data.sort_values('time')
daily_data.set_index('time', inplace=True)
daily_data['gradient'] = daily_data['close'].diff().fillna(0.0)
daily_data

In [ ]:
daily_data['log_ret'] = np.log(daily_data['close'] / daily_data['close'].shift(1))
daily_data = daily_data.dropna(subset=['log_ret'])

# Annualize drift (mu) and volatility (sigma)
steps_per_year = 365
mu_hat    = daily_data['log_ret'].mean() * steps_per_year
sigma_hat = daily_data['log_ret'].std(ddof=0) * np.sqrt(steps_per_year)

print(f"Estimated annual drift (mu): {mu_hat:.4f}")
print(f"Estimated annual vol   (sigma): {sigma_hat:.4f}")


In [ ]:
def gbm_paths_numpy(S0, mu, sigma, T=1.0, steps_per_year=365, n_paths=5, seed=None):
    """
    Simulate GBM sample paths with explicit Euler scheme.
    Returns a DataFrame of shape (N+1) x n_paths, indexed by pandas dates.
    """
    if seed is not None:
        np.random.seed(seed)

    dt = 1 / steps_per_year
    N  = int(T * steps_per_year)
    # generate random shocks
    Z = np.random.normal(size=(N, n_paths))
    # log-return increments
    increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    increments = np.vstack([np.zeros(n_paths), increments])
    # price paths
    S = S0 * np.exp(np.cumsum(increments, axis=0))

    # build date index starting at today
    start = df.index[-1]  # last date in your historical series
    dates = pd.date_range(start=start, periods=N+1, freq='D')
    return pd.DataFrame(S, index=dates)